# HW/SW Co-Verification: Piecewise Sigmoid Accelerator
This notebook validates the hardware accelerator against the software model and visualizes the error.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Add local python modules
sys.path.append(os.path.abspath('../driver'))
sys.path.append(os.path.abspath('../../model'))

from accel_driver import SigmoidAccelerator
from float_model import golden_function

In [ ]:
# Initialize the Hardware Overlay
try:
    acc = SigmoidAccelerator('../../overlay/design.bit')
    print('Hardware Overlay Loaded Successfully!')
except Exception as e:
    print(f'Error loading hardware overlay: {e}')
    acc = None

In [ ]:
# Generate Test Vector Array
x_vals = np.linspace(-6.0, 6.0, 100)

# Run Software Golden Model
y_sw = golden_function(x_vals)

# Run Hardware Accelerator (Pipelined Execution)
if acc is not None:
    y_hw = acc.compute_batch(x_vals)
else:
    print('Hardware not loaded. Using dummy data for demonstration.')
    # Placeholder fallback for local testing
    y_hw = y_sw + np.random.normal(0, 0.01, size=len(x_vals))

# Calculate Error
y_hw = np.array(y_hw)
error = np.abs(y_sw - y_hw)
max_error = np.max(error)
print(f'Max Absolute Error: {max_error:.6f}')

In [ ]:
# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Sigmoid Curves
ax1.plot(x_vals, y_sw, label='Software (Golden)', linewidth=3, alpha=0.7)
ax1.plot(x_vals, y_hw, '--', label='Hardware (Q4.12 PWL)', color='red')
ax1.set_title('Sigmoid: HW vs SW')
ax1.set_xlabel('Input x')
ax1.set_ylabel('Output y')
ax1.legend()
ax1.grid(True)

# Plot 2: Error
ax2.plot(x_vals, error, label='Absolute Error', color='orange')
ax2.axhline(y=0.02, color='r', linestyle=':', label='0.02 Threshold')
ax2.set_title(f'Quantization Error (Max: {max_error:.4f})')
ax2.set_xlabel('Input x')
ax2.set_ylabel('Error magnitude')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()